In [ ]:
import json
import syncedlyrics
from mutagen.flac import FLAC
from mutagen.mp4 import MP4
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

MP4_TAG_MAP = {"title": "©nam", "artist": "©ART", "album": "©alb"}

In [ ]:
def get_flac_metadata(filepath: str) -> dict:
    audio = FLAC(filepath)
    return {k: (audio.get(k) or [""])[0].strip() for k in ("title", "artist", "album")}

def get_alac_metadata(filepath: str) -> dict:
    audio = MP4(filepath)
    return {
        k: (audio.tags.get(tag) or [""])[0].strip()
        for k, tag in MP4_TAG_MAP.items()
    }

def get_metadata(filepath: Path) -> dict | None:
    try:
        if filepath.suffix.lower() == ".flac":
            return get_flac_metadata(str(filepath))
        elif filepath.suffix.lower() == ".m4a":
            return get_alac_metadata(str(filepath))
    except Exception as e:
        print(f"  Ошибка метаданных {filepath.name}: {e}")
    return None

def get_lyrics(title: str, artist: str) -> str | None:
    try:
        return syncedlyrics.search(
            f"{title} {artist}",
            providers=["Lrclib", "NetEase", "Megalobiz"],
            plain_only = True
        )
    except Exception:
        return None

def process_file(filepath: Path) -> dict | None:
    meta = get_metadata(filepath)
    if not meta or not meta["title"] or not meta["artist"]:
        print(f"  Пропуск: {filepath.name}")
        return None

    lyrics = get_lyrics(meta["title"], meta["artist"])
    print(f"{'✓' if lyrics else '✗'} {meta['artist']} — {meta['title']}")
    return {**meta, "lyrics": lyrics}

def fetch_lyrics_bulk(folder: str, output_file: str = "lyrics.json", workers: int = 8):
    audio_files = [
        p for p in Path(folder).rglob("*")
        if p.suffix.lower() in (".flac", ".m4a")
    ]
    print(f"Найдено файлов: {len(audio_files)}, воркеров: {workers}")

    results = {}
    with ThreadPoolExecutor(max_workers=workers) as executor:
        futures = {executor.submit(process_file, f): f for f in audio_files}
        for future in as_completed(futures):
            meta = future.result()
            if meta:
                key = f"{meta['artist']} — {meta['title']}"
                results.setdefault(key, meta)

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    found = sum(1 for v in results.values() if v["lyrics"])
    print(f"\nГотово: {found}/{len(results)} текстов найдено → {output_file}")
    return results



In [ ]:
folder = "S:\Music"
fetch_lyrics_bulk(folder=folder)

In [ ]:
[(k, v.get('lyrics').splitlines()) for k, v in data.items()]